## 라이브러리 임포트

분석에 필요한 라이브러리를 불러옵니다.

In [1]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

## 데이터 로드 및 파생 컬럼 생성

`steam_indie_list.csv` 원본 데이터를 불러오고, 분석에 필요한 파생 컬럼을 생성합니다.

- `total_reviews`: 긍정 + 부정 리뷰 합산
- `owners_lower`: `owners` 범위 문자열에서 하한값 추출
- `is_f2p`: 무료 플레이 여부
- `is_early_access`: 얼리 액세스 여부

In [2]:
df = pd.read_csv('../../../data/raw/steam_indie_list.csv')

df['total_reviews'] = df['positive'] + df['negative']
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')

def parse_owners_lower(s):
    try:
        return int(s.split('..')[0].strip().replace(',', ''))
    except Exception:
        return 0

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except Exception:
        return []

df['owners_lower']    = df['owners'].apply(parse_owners_lower)
df['genres']          = df['genres'].apply(parse_genres)
df['is_f2p']          = df['genres'].apply(lambda gl: 'Free To Play' in gl)
df['is_early_access'] = df['genres'].apply(lambda gl: 'Early Access' in gl)

print(f'원본: {len(df):,}개')
print(f'Early Access: {df["is_early_access"].sum():,}개 ({df["is_early_access"].mean():.1%})')
print(f'F2P: {df["is_f2p"].sum():,}개 ({df["is_f2p"].mean():.1%})')

원본: 61,266개
Early Access: 6,340개 (10.3%)
F2P: 3,446개 (5.6%)


## 분석 대상 필터링

다음 조건을 모두 만족하는 게임을 메인 분석 모집단으로 선별합니다.

- 출시연도: 2023 ~ 2025년
- 리뷰 수: 10개 이상
- Early Access 제외
- Free to Play 제외

Early Access와 F2P 게임은 별도 데이터프레임(`df_ea`, `df_f2p`)으로 보존합니다.

In [ ]:
MIN_REVIEWS = 10

df_ea  = df[df['is_early_access']].copy()           # Early Access 별도 보관
df_f2p = df[~df['is_early_access'] & df['is_f2p']].copy()  # F2P 별도 보관
df_f   = df[
    (df['total_reviews'] >= MIN_REVIEWS) &
    (df['release_date'].dt.year >= 2023) &
    (df['release_date'].dt.year <= 2025) &
    (~df['is_early_access']) &
    (~df['is_f2p'])
].copy()

print(f'전체              : {len(df):,}개')
print(f'Early Access 제외 : {len(df_ea):,}개 → 별도 분석')
print(f'F2P 제외          : {len(df_f2p):,}개 → 별도 분석')
print(f'메인 모집단       : {len(df_f):,}개  (2023~2025년, 리뷰 {MIN_REVIEWS}개 이상, EA·F2P 제외)')
print(f'\n출시연도 분포 (메인):')
print(df_f['release_date'].dt.year.value_counts().sort_index().to_string())

전체              : 61,266개
Early Access 제외 : 6,340개 → 별도 분석
F2P 제외          : 3,064개 → 별도 분석
메인 모집단       : 9,692개  (2023~2025년, 리뷰 10개 이상, EA·F2P 제외)

출시연도 분포 (메인):
release_date
2023    3498
2024    4180
2025    2014


## 컬럼 정제

분석에 적합한 형태로 컬럼명을 정리하고 불필요한 컬럼을 제거합니다.

- `name`: `name_store` 우선, 없으면 `spy_name` 사용
- `price`: `price_spy` 컬럼명 변경
- `release_date`: `yyyy-MM-dd` 포맷으로 통일
- 불필요 컬럼(`spy_name`, `name_store`) 제거

In [ ]:
# ── name_store 값으로 name_spy 대체 후 name 으로 컬럼명 변경 ─────────────────
df_f['name'] = df_f['name_store'].fillna(df_f['spy_name'])

# ── price_spy → price 컬럼명 변경 ────────────────────────────────────────────
df_f = df_f.rename(columns={'price_spy': 'price'})

# ── release_date → yyyy-MM-dd 포맷 ───────────────────────────────────────────
df_f['release_date'] = df_f['release_date'].dt.strftime('%Y-%m-%d')

# ── 불필요 컬럼 제거 ──────────────────────────────────────────────────────────
df_f = df_f.drop(columns=['spy_name', 'name_store', 'type'])

print('전처리 완료')
print(df_f[['name', 'release_date', 'price']].head())

전처리 완료
                              name release_date  price
10                      Last Epoch   2024-02-21   3499
23                   7 Days to Die   2024-07-25   4499
24                       CyberCorp   2025-04-22   1499
25              Sons Of The Forest   2024-02-22   2999
28  Warhammer 40,000: Rogue Trader   2023-12-07   4999


## 데이터 저장

전처리가 완료된 메인 모집단을 `data/processed/steam_indie_9692.csv` 파일로 저장합니다.

In [5]:
out_path = '../../../data/processed/steam_indie_9692.csv'
df_f.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_f):,}개)')

저장 완료 → ../../../data/processed/steam_indie_9692.csv (9,692개)
